In [1]:
import os
import shutil

# 1. Retreat to base and clean
os.chdir('/kaggle/working')
repo_path = '/kaggle/working/Patch-DM'
if os.path.exists(repo_path):
    print("🧹 Cleaning up old repository...")
    shutil.rmtree(repo_path)

# 2. Fresh Clone
!git clone https://github.com/mlpc-ucsd/Patch-DM.git
%cd Patch-DM

# 3. Dependencies
!pip install -q pytorch-lightning==1.9.5 lmdb==1.2.1 lpips pytorch-fid einops
!pip install -q git+https://github.com/openai/CLIP.git

print("\n✅ Environment clean and ready.")

Cloning into 'Patch-DM'...
remote: Enumerating objects: 93, done.
remote: Counting objects: 100% (93/93), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 93 (delta 21), reused 85 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (93/93), 1.06 MiB | 8.24 MiB/s, done.
Resolving deltas: 100% (21/21), done.
/kaggle/working/Patch-DM
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 881.5/881.5 kB 15.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.5/829.5 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.8 MB/s eta 0:00:00

✅ Environment clean and ready.


In [2]:
import os
import re

# --- 1. Fix model/unet.py (Novelty Injection) ---
with open('model/unet.py', 'r') as f:
    unet_lines = f.readlines()

# Define the blender class
blender_class = [
    "\nclass GatedWeightBlender(nn.Module):\n",
    "    def __init__(self, channels):\n",
    "        super().__init__()\n",
    "        self.gate = nn.Sequential(\n",
    "            nn.Conv2d(channels, channels // 2, kernel_size=1),\n",
    "            nn.SiLU(),\n",
    "            nn.Conv2d(channels // 2, 1, kernel_size=1),\n",
    "            nn.Sigmoid()\n",
    "        )\n",
    "    def forward(self, x_orig, x_shifted):\n",
    "        alpha = self.gate(x_orig)\n",
    "        return alpha * x_orig + (1 - alpha) * x_shifted\n\n"
]
# Insert class after imports
unet_lines.insert(15, "".join(blender_class))
unet_content = "".join(unet_lines)

# Inject initialization into __init__ (ONLY ONCE before the if/else block)
# Anchor: the comment before the output layer block
init_anchor = "if conf.resnet_use_zero_module:"
unet_content = unet_content.replace(init_anchor, f"self.blender = GatedWeightBlender(ch)\n        {init_anchor}", 1)

# Inject blending into forward pass
forward_anchor = "pred = self.out(h)"
blending_logic = "h_blended = self.blender(h, torch.roll(h, shifts=(2,2), dims=(2,3)))\n        pred = self.out(h_blended)"
unet_content = unet_content.replace(forward_anchor, blending_logic, 1)

with open('model/unet.py', 'w') as f:
    f.write(unet_content)

# --- 2. Fix experiment.py (Stability & Logger) ---
with open('experiment.py', 'r') as f:
    exp_content = f.read()

# Fix Numpy Import
exp_content = exp_content.replace('from numpy.lib.function_base import flip', 'from numpy import flip')

# Fix Hook Signature
exp_content = re.sub(
    r'def on_train_batch_end\s*\([^)]+\)\s*->\s*None:',
    r'def on_train_batch_end(self, outputs, batch, batch_idx: int) -> None:',
    exp_content
)

# Fix Trainer Timeouts
exp_content = exp_content.replace("desc='infer')", "desc='infer', disable=True)")
exp_content = exp_content.replace('trainer = pl.Trainer(', 'trainer = pl.Trainer(enable_progress_bar=False, max_time="00:11:30:00", ')

# Define and Append Logger at the end to avoid import errors
logger_code = """
class ConsoleLossLogger(pl.Callback):
    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        if batch_idx % 1000 == 0:
            loss = outputs['loss'].item()
            print(f"   [Step {trainer.global_step}] Loss: {loss:.4f} | Epoch: {trainer.current_epoch}")
"""
exp_content += "\n" + logger_code

# Inject Logger into the train function's callback list
exp_content = exp_content.replace('callbacks = [LearningRateMonitor()]', 'callbacks = [LearningRateMonitor(), ConsoleLossLogger()]')

with open('experiment.py', 'w') as f:
    f.write(exp_content)

# --- 3. Fix train.py (Frequency) ---
with open('train.py', 'r') as f:
    train_content = f.read()
train_content = train_content.replace('conf.save_every_samples = 100_000', 'conf.save_every_samples = 10_000')
with open('train.py', 'w') as f:
    f.write(train_content)

print("🚀 Precision patches applied. No duplicate strings detected.")

🚀 Precision patches applied. No duplicate strings detected.


In [3]:
import torch

# Paths
INPUT_SEMANTIC = "/kaggle/input/datasets/vanditagupta2609/lmdb-input-2/lmdb-input/semantic.pt"
FIXED_SEMANTIC = "/kaggle/working/semantic_fixed.pt"
DATA_PATH = "/kaggle/input/datasets/vanditagupta2609/lmdb-input-2/lmdb-input/ffhq256.lmdb"
CHECKPOINT_DIR = '/kaggle/working/Patch-DM/checkpoints/patchdm_novelty/'

if os.path.exists(INPUT_SEMANTIC):
    data = torch.load(INPUT_SEMANTIC, map_location='cpu')
    if isinstance(data, dict) and 'model_state_dict' not in data:
        key = list(data.keys())[0]
        wrapped = {'model_state_dict': {'semantic_enc.weight': data[key]}}
    elif torch.is_tensor(data):
        wrapped = {'model_state_dict': {'semantic_enc.weight': data}}
    else:
        wrapped = data
    torch.save(wrapped, FIXED_SEMANTIC)
    print(f"✅ Semantic file ready: {FIXED_SEMANTIC}")

✅ Semantic file ready: /kaggle/working/semantic_fixed.pt


In [4]:
import subprocess
import time
import os

CHECKPOINT_DIR = '/kaggle/working/Patch-DM/checkpoints/patchdm_novelty/'

train_cmd = [
    "python", "-u", "train.py",
    "--batch_size", "4",
    "--patch_size", "64",
    "--data_path", DATA_PATH,
    "--name", "patchdm_novelty",
    "--semantic_path", FIXED_SEMANTIC
]

print("🔥 Launching Gated Patch-DM (Real-time updates below)...")
# Using bufsize=1 for line-buffered streaming
process = subprocess.Popen(train_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

last_cleanup = time.time()

try:
    for line in process.stdout:
        print(line, end="")
        
        # Disk Guardian (10 min check)
        if time.time() - last_cleanup > 600:
            if os.path.exists(CHECKPOINT_DIR):
                files = [os.path.join(CHECKPOINT_DIR, f) for f in os.listdir(CHECKPOINT_DIR) if f.endswith('.ckpt')]
                if len(files) > 2:
                    files.sort(key=os.path.getmtime)
                    for f in files[:-2]: 
                        os.remove(f)
                        print(f"\n🛡️ Disk Guardian: Removed {os.path.basename(f)}")
            last_cleanup = time.time()
except KeyboardInterrupt:
    print("\n🛑 Manual Termination...")
    process.terminate()

🔥 Launching Gated Patch-DM (Real-time updates below)...
conf: patchdm_novelty
Global seed set to 0
Model params: 102.28 M
==Model size is 64==
ModelCheckpoint(save_last=True, save_top_k=-1, monitor=None) will duplicate the last checkpoint saved.
ckpt path: checkpoints/patchdm_novelty/last.ckpt
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/connectors/accelerator_connector.py:478: LightningDeprecationWarning: Setting `Trainer(gpus=[0])` is deprecated in v1.7 and will be removed in v2.0. Please use `Trainer(accelerator='gpu', devices=[0])` instead.
  rank_zero_deprecation(
Using 16bit None Automatic Mixed Precision (AMP)
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/plugins/precision/native_amp.py:47: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available:

In [5]:
print("📦 Packaging results...")
%cd /kaggle/working/
!zip -r /kaggle/working/patchdm_novelty_final.zip /kaggle/working/Patch-DM/checkpoints/patchdm_novelty/
print("✅ Done.")

📦 Packaging results...
/kaggle/working
  adding: kaggle/working/Patch-DM/checkpoints/patchdm_novelty/ (stored 0%)
  adding: kaggle/working/Patch-DM/checkpoints/patchdm_novelty/hparams.yaml (deflated 65%)
  adding: kaggle/working/Patch-DM/checkpoints/patchdm_novelty/sample/ (stored 0%)
  adding: kaggle/working/Patch-DM/checkpoints/patchdm_novelty/sample/500000.png (deflated 0%)
  adding: kaggle/working/Patch-DM/checkpoints/patchdm_novelty/epoch=7-step=127500.ckpt (deflated 14%)
  adding: kaggle/working/Patch-DM/checkpoints/patchdm_novelty/events.out.tfevents.1774635580.497b007564fd.108.0 (deflated 61%)
  adding: kaggle/working/Patch-DM/checkpoints/patchdm_novelty/last.ckpt (deflated 14%)
✅ Done.
